In [1]:
import os
import json
import requests
import time

BASE_DIR = "../JCDL_Code_2022_2025/Scientific_Novelty_Detection_2022_2025"
CACHE_DIR = os.path.join(BASE_DIR, "cache")
CHECKPOINT_DIR = os.path.join(BASE_DIR, "checkpoints")

os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

YEAR_START = 2022
YEAR_END = 2025
PER_PAGE = 200
TOP_K = 120

TASKS = {
    "Dia": "dialogue",
    "MT": "translation",
    "NLI": "natural language inference",
    "Par": "paraphrase",
    "QA": "question answering",
    "SA": "sentiment",
    "Sum": "summarization"
}

In [2]:
def fetch_blogs_task(task, keyword):

    cache_file = os.path.join(CACHE_DIR, f"Blogs_{task}_metadata.json")
    checkpoint_file = os.path.join(CHECKPOINT_DIR, f"Blogs_{task}_fetch_checkpoint.json")

    if os.path.exists(cache_file):
        print(f"Loading cached Blogs for {task}")
        with open(cache_file, "r") as f:
            return json.load(f)

    print(f"\nFetching Blogs for {task}")

    if os.path.exists(checkpoint_file):
        with open(checkpoint_file, "r") as f:
            checkpoint = json.load(f)
        cursor = checkpoint["cursor"]
        papers = checkpoint["papers"]
        print(f"Resuming from cursor {cursor}")
    else:
        cursor = "*"
        papers = []

    while True:

        params = {
            "search": keyword,
            "filter": f"publication_year:{YEAR_START}-{YEAR_END}",
            "sort": "cited_by_count:desc",
            "per-page": PER_PAGE,
            "cursor": cursor,
            "mailto": "your_email@example.com"
        }

        r = requests.get("https://api.openalex.org/works", params=params, timeout=30)

        if r.status_code != 200:
            print("API error:", r.status_code)
            break

        data = r.json()
        results = data.get("results", [])

        if not results:
            break

        for paper in results:

            # Exclude peer-reviewed backbone
            if paper.get("type") in ["article", "proceedings-article"]:
                continue

            papers.append({
                "id": paper.get("id"),
                "title": paper.get("title"),
                "year": paper.get("publication_year"),
                "pdf_url": paper.get("open_access", {}).get("oa_url")
            })

        print(f"{task} — Collected {len(papers)}")

        with open(checkpoint_file, "w") as f:
            json.dump({
                "cursor": data["meta"]["next_cursor"],
                "papers": papers
            }, f, indent=2)

        if len(papers) >= TOP_K:
            break

        cursor = data["meta"]["next_cursor"]

        if not cursor:
            break

        time.sleep(1)

    papers = papers[:TOP_K]

    with open(cache_file, "w") as f:
        json.dump(papers, f, indent=2)

    if os.path.exists(checkpoint_file):
        os.remove(checkpoint_file)

    print(f"{task} Blogs final count: {len(papers)}")

    return papers

In [3]:
for task, keyword in TASKS.items():
    fetch_blogs_task(task, keyword)

print("All Blogs metadata fetched successfully.")


Fetching Blogs for Dia
Dia — Collected 72
Dia — Collected 151
Dia Blogs final count: 120

Fetching Blogs for MT
MT — Collected 99
MT — Collected 221
MT Blogs final count: 120

Fetching Blogs for NLI
NLI — Collected 60
NLI — Collected 123
NLI Blogs final count: 120

Fetching Blogs for Par
Par — Collected 52
Par — Collected 93
Par — Collected 126
Par Blogs final count: 120

Fetching Blogs for QA
QA — Collected 78
QA — Collected 144
QA Blogs final count: 120

Fetching Blogs for SA
SA — Collected 73
SA — Collected 137
SA Blogs final count: 120

Fetching Blogs for Sum
Sum — Collected 96
Sum — Collected 205
Sum Blogs final count: 120
All Blogs metadata fetched successfully.
